# 🟠 PR Score — Automação Loft

**Como usar:**
1. **Célula 1** — Instala dependências (rode uma vez por sessão)
2. **Célula 2** — Configure a chave da API Claude
3. **Célula 3** — Carrega a lista Tier 1
4. **Célula 4** — Upload e limpeza da planilha da Clipadora
5. **Célula 5** — Claude avalia protagonismo matéria por matéria
6. **Célula 6** — Gera planilha .xlsx processada para download
7. **Célula 7** — Gera texto do WhatsApp e HTML do e-mail

---
⚠️ **Pré-requisito:** chave da API Claude em [console.anthropic.com](https://console.anthropic.com) → API Keys

In [ ]:
# ============================================================
# CÉLULA 1 — Instalar dependências
# ============================================================
!pip install anthropic openpyxl requests beautifulsoup4 --quiet

import anthropic
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import requests
import json
import re
import time
from datetime import date
from bs4 import BeautifulSoup
from google.colab import files
from io import BytesIO

print('✅ Dependências instaladas com sucesso!')

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração
# ============================================================

# Chave da API Claude (obter em console.anthropic.com → API Keys)
ANTHROPIC_API_KEY = ""  # ← Cole sua chave aqui

# Modelo Claude a usar
CLAUDE_MODEL = "claude-sonnet-4-6"

# Data do clipping (padrão: hoje)
DATA_CLIPPING = date.today().strftime("%d/%m/%Y")
# Para outra data, descomente:
# DATA_CLIPPING = "25/06/2026"

assert ANTHROPIC_API_KEY, "❌ Configure o ANTHROPIC_API_KEY!"

# Inicializar cliente
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Teste rápido
teste = client.messages.create(
    model=CLAUDE_MODEL, max_tokens=10,
    messages=[{"role": "user", "content": "Responda apenas: OK"}]
)
print(f"✅ Claude conectado! Resposta: {teste.content[0].text.strip()}")
print(f"✅ Configuração OK — clipping de {DATA_CLIPPING}")

In [ ]:
# ============================================================
# CÉLULA 3 — Carregar lista Tier 1
# ============================================================

print("📋 Faça upload da planilha Tier 1 (Hackaton_Tier_1__Loft_1.xlsx):")
uploaded_tier1 = files.upload()
tier1_filename = list(uploaded_tier1.keys())[0]

df_t1 = pd.read_excel(BytesIO(uploaded_tier1[tier1_filename]), header=None)
TIER1 = set()
for col in df_t1.columns:
    for val in df_t1[col].dropna():
        v = str(val).strip()
        if v:
            TIER1.add(v.lower())

print(f"✅ {len(TIER1)} veículos Tier 1 carregados")

# Constantes
PAYWALL_VEICULOS   = ["valor econômico", "jornal do comércio"]
CANAIS_EXCLUIR     = ["coelho da fonseca", "lopes", "assuntos de interesse"]
TEMAS_BASE = [
    "Mercado Imobiliário", "Precificação", "Locação", "Compra e Venda",
    "Tecnologia e Produto", "Porta-voz", "Pesquisas e Tendências",
    "Expansão e Negócios", "Crise e Reputação", "Mercado de Capitais",
    "Urbanismo e Cidades", "Financiamento e Crédito", "ESG", "RH e Cultura",
    "Transações", "Crédito Imobiliário"
]

# Fills para colorir células
FILL_VERMELHO  = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
FILL_AMARELO   = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
FILL_HEADER    = PatternFill(start_color="FF6B35", end_color="FF6B35", fill_type="solid")
FONT_HEADER    = Font(bold=True, color="FFFFFF")

def normalizar_empresa(canal):
    c = str(canal or "").lower()
    if "loft" in c or "foxter" in c:            return "Loft"
    if "zap" in c or "olx" in c or "viva real" in c: return "ZAP"
    if "quintoandar" in c or "quinto andar" in c or "imovelweb" in c: return "QuintoAndar"
    if "superlógica" in c or "superlogica" in c: return "Superlógica"
    if "creditas" in c:                          return "Creditas"
    if "porto seguro" in c:                      return "Porto Seguro"
    return str(canal or "")

def is_tier1(v):    return str(v or "").strip().lower() in TIER1
def is_paywall(v):  return any(p in str(v or "").lower() for p in PAYWALL_VEICULOS)
def is_excluido(c): return any(e in str(c or "").lower() for e in CANAIS_EXCLUIR)

print("✅ Pronto para a Célula 4!")

In [ ]:
# ============================================================
# CÉLULA 4 — Upload e limpeza da planilha da Clipadora
# ============================================================

print("📂 Faça upload da planilha da Clipadora (.xlsx):")
uploaded = files.upload()
clip_filename = list(uploaded.keys())[0]

# Headers na linha 4 (índice 3)
df = pd.read_excel(BytesIO(uploaded[clip_filename]), sheet_name="Matérias", header=3)
total_original = len(df)

# Renomear colunas pelo índice (mais robusto que pelo nome)
# A=0 título, B=1 veículo, C=2 data, G=6 canal, L=11 estado, M=12 mídia, W=22 release, AB=27 link
cols = df.columns.tolist()
def col(i): return cols[i] if i < len(cols) else None

df = df.rename(columns={
    col(0):  "titulo",
    col(1):  "veiculo",
    col(2):  "data_raw",
    col(6):  "canal",
    col(11): "estado",
    col(12): "midia",
    col(26): "url_fonte",   # coluna AA (índice 26)
    col(22): "release",
    col(23): "mat_repetida",
    col(27): "link"
}).copy()

df = df[df["titulo"].notna() & (df["titulo"].astype(str).str.strip() != "")].copy()
print(f"📊 {total_original} matérias no clipping original")

# Corrigir data
def conv_data(v):
    if pd.isna(v): return DATA_CLIPPING
    if isinstance(v, (int, float)):
        try:
            from datetime import datetime, timedelta
            dt = datetime(1899,12,30) + timedelta(days=float(v))
            return dt.strftime("%d/%m/%Y")
        except: pass
    try:
        return pd.to_datetime(v, dayfirst=True).strftime("%d/%m/%Y")
    except:
        return str(v)

df["data"] = df["data_raw"].apply(conv_data)
df["mes"]  = df["data"].apply(lambda d: d.split("/")[1].lstrip("0") if "/" in str(d) else "")
df["empresa"] = df["canal"].apply(normalizar_empresa)

# Motivo para pintar de vermelho
def motivo_vermelho(row):
    if is_excluido(row.get("canal", "")):
        return "Canal excluído"
    if not is_tier1(row.get("veiculo", "")):
        return "Não é Tier 1"
    r = str(row.get("release", "") or "").lower()
    if r in ("sim", "yes", "true", "1"):
        return "Press release pago"
    return ""

df["motivo_vermelho"] = df.apply(motivo_vermelho, axis=1)

n_vermelho = (df["motivo_vermelho"] != "").sum()
df_validas  = df[df["motivo_vermelho"] == ""].reset_index(drop=True)
df_vermelhas = df[df["motivo_vermelho"] != ""]

# Matérias off-topic → Double Check
off_kw = ["carro", "automóvel", "celular", "smartphone", "veículo automotor"]
mask_off = df_validas["titulo"].apply(lambda t: any(k in str(t).lower() for k in off_kw))
df_double_check = df_validas[mask_off].copy()
df_main = df_validas[~mask_off].reset_index(drop=True)

print(f"\n📋 RESUMO DA LIMPEZA")
print(f"   Total original:       {total_original}")
print(f"   Pintadas de vermelho: {n_vermelho}")
print(f"   Para avaliação:       {len(df_main)}")
print(f"   Double Check:         {len(df_double_check)}")

In [ ]:
# ============================================================
# CÉLULA 5 — Avaliação de protagonismo com Claude
# ============================================================

PROMPT_AVALIACAO = (
    "Você é um especialista em avaliação de PR Score para empresas do mercado imobiliário brasileiro.\n"
    "\n"
    "Sua tarefa é avaliar a matéria a seguir e preencher todos os campos solicitados.\n"
    "\n"
    "EMPRESA MONITORADA NESTA MATÉRIA: {empresa}\n"
    "TÍTULO: {titulo}\n"
    "VEÍCULO: {veiculo}\n"
    "TIPO DE MÍDIA: {tipo_midia}\n"
    "CONTEÚDO DA MATÉRIA:\n"
    "{conteudo}\n"
    "\n"
    "---\n"
    "\n"
    "## FILTRO INICIAL (aplicar primeiro)\n"
    "\n"
    "Verifique se a matéria deve ser excluída:\n"
    "- Está em aba de 'Releases Empresariais' explícita: EXCLUIR\n"
    "- É conteúdo 'Patrocinado' ou 'Conteúdo de Marca' não integrado ao editorial: EXCLUIR\n"
    "- É sobre segmento fora do core imobiliário da empresa: EXCLUIR\n"
    "- EXCEÇÃO: releases de terceiros que citam a marca organicamente: MANTER\n"
    "\n"
    "Se excluída, retornar protagonismo = 'Excluído' e explicar em obs.\n"
    "\n"
    "---\n"
    "\n"
    "## PARTE 2 — CRITÉRIOS PRINCIPAIS (não-cumulativos, gatilho para pontuar)\n"
    "\n"
    "Verifique se PELO MENOS UM dos critérios abaixo é atendido.\n"
    "Se nenhum for atendido: protagonismo = 'Menção', pontuação = 0, encerrar.\n"
    "Se algum for atendido: protagonismo = 'Destaque', pontuação começa em 1.\n"
    "\n"
    "1. Porta-voz Ativo ou Artigo Assinado: 3+ frases de porta-voz oficial com aspas diretas, OU texto integralmente assinado pelo porta-voz.\n"
    "2. Espaço Qualificado: 5+ frases completas OU 8+ linhas somadas ao longo do texto fazendo referência à marca/porta-voz.\n"
    "3. Densidade de Exposição: Trechos da marca ocupam 30%+ do volume total da matéria.\n"
    "4. Fontes de Dados: Marca explicitamente citada como fonte dos dados/pesquisas, ou porta-voz conduz análise técnica dos dados.\n"
    "5. Colunas Curtas: Nota com até 300 caracteres onde a marca é protagonista. Se passar de 300, avaliar pelos critérios 1-4.\n"
    "6. Lives: 1.000+ visualizações. [SINALIZAR COMO HUMANO — critério 6]\n"
    "7. Anúncios Governamentais: Citação em anúncio de governo federal, estadual ou prefeitura de capital.\n"
    "\n"
    "---\n"
    "\n"
    "## PARTE 3 — CRITÉRIOS EXTRAS (cumulativos, +1 ponto cada)\n"
    "\n"
    "Aplicar apenas se a matéria já garantiu o ponto inicial.\n"
    "\n"
    "8. Protagonismo Editorial: +1 por cada local onde a marca aparece no bloco de abertura: título (+1), chapéu (+1), subtítulo (+1), linha fina (+1). Máx. +4.\n"
    "9. Volume de Menções: 6+ menções nominais à marca no texto: +1.\n"
    "10. CTA/Link: Chamadas 'Leia mais' ou 'Veja também' direcionando para matéria da marca: +1 por ocorrência. Hiperlinks fluidos em palavras soltas NÃO contam.\n"
    "11. Recursos Visuais: +1 por TIPO DIFERENTE de imagem: foto-legenda, foto-logo, foto de executivo. Repetições do mesmo tipo = apenas +1. [SINALIZAR QUANDO NÃO CONFIRMÁVEL PELO TEXTO]\n"
    "12. Gráficos/Tabelas/Elementos: +1 por TIPO DIFERENTE: tabelas, gráficos, boxes, olho, listas. Repetições = apenas +1. [SINALIZAR QUANDO NÃO CONFIRMÁVEL PELO TEXTO]\n"
    "13. Páginas Impressas: +1 por página física adicional com destaque/artigo da marca (só jornais e revistas físicas).\n"
    "14. Capas e Redes Sociais: [SEMPRE SINALIZAR COMO HUMANO — critério 14]\n"
    "    - Feed do veículo (Instagram, LinkedIn, Threads, X, Facebook) com chamada dedicada: +1 por ocorrência\n"
    "    - Chamada secundária na capa/home page: +1 por ocorrência\n"
    "    - Manchete principal: +1 sem citar nome (por contexto); +2 se o nome da marca aparecer explicitamente\n"
    "15. Audiovisual (Rádio, TV, Podcast): [SEMPRE SINALIZAR COMO HUMANO — critério 15]\n"
    "    - +1 a cada 2 minutos após os 2 primeiros de exposição\n"
    "    - +1 por intervenção de voz (sonora) do porta-voz\n"
    "    - +1 por tipo de recurso visual diferente em tela (tarja, tabela, gráfico)\n"
    "    - Teto: 4 pontos por subtipo em regionais; 8 em nacionais\n"
    "\n"
    "---\n"
    "\n"
    "## PARTE 4 — ANÁLISE ESPECIAL (matérias longas)\n"
    "\n"
    "16. Protocolo de 10+ Trechos: [SEMPRE SINALIZAR COMO HUMANO — critério 16]\n"
    "    - Ativado se a matéria tiver 10+ trechos/análises profundas da marca ocupando o lead\n"
    "    - A cada 5 sentenças positivas: +1; a cada 5 negativas: -1\n"
    "\n"
    "---\n"
    "\n"
    "Responda EXCLUSIVAMENTE em JSON com o seguinte formato (sem texto antes ou depois):\n"
    "\n"
    '{{\n'
    '  "protagonismo": "Destaque | Menção | Destaque Negativo | Menção Negativa",\n'
    '  "criterio_ativado": "número de 1 a 7 que ativou o protagonismo, ou nenhum",\n'
    '  "criterio_8": 0,\n'
    '  "criterio_9": 0,\n'
    '  "criterio_10": 0,\n'
    '  "criterio_11": 0,\n'
    '  "criterio_12": 0,\n'
    '  "criterio_13": 0,\n'
    '  "criterio_14": "ver humano",\n'
    '  "criterio_15": 0,\n'
    '  "criterio_16": "ver humano",\n'
    '  "soma": 0,\n'
    '  "pr_tec_produto": false,\n'
    '  "pr_puro_imobis": false,\n'
    '  "data": true | false,\n'
    '  "data_carona": false,\n'
    '  "tema": "",\n'
    '  "subtema": "",\n'
    '  "produto": "",\n'
    '  "obs": "",\n'
    '  "cidade": "",\n'
    '  "confianca": "alta"\n'
    "}}\n"
    "\n"
    "Regras obrigatórias:\n"
    "- pr_tec_produto e pr_puro_imobis são SEMPRE false para empresas que não são Loft\n"
    "- criterio_14 e criterio_16 são SEMPRE 'ver humano'\n"
    "- criterio_15 é 'ver humano' se a matéria for de rádio, TV ou podcast; caso contrário 0\n"
    "- criterio_11 e criterio_12 são 'ver humano' quando não for possível confirmar pelo texto\n"
    "- soma = 1 (se Destaque) + soma numérica dos critérios 8 a 16 (ignora 'ver humano')\n"
    "- Se paywall impedir leitura: obs deve registrar 'PAYWALL — leitura incompleta' e confianca = 'baixa'\n"
    "- Sentenças mistas contraditórias: protagonismo = 'Destaque Negativo' e explicar em obs"
)


def raspar_texto(url):
    if not str(url or "").startswith("http"):
        return ""
    try:
        r = requests.get(str(url), headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer"]):
            t.decompose()
        return " ".join(p.get_text().strip() for p in soup.find_all("p"))[:4500]
    except:
        return ""


def _vazio_avaliacao(obs_msg=""):
    return {
        "protagonismo": "Menção", "criterio_ativado": "nenhum",
        "criterio_8": 0, "criterio_9": 0, "criterio_10": 0,
        "criterio_11": 0, "criterio_12": 0, "criterio_13": 0,
        "criterio_14": "ver humano", "criterio_15": 0, "criterio_16": "ver humano",
        "soma": 0, "pr_tec_produto": False, "pr_puro_imobis": False,
        "dado_proativo": False, "data_carona": False,
        "tema": "", "subtema": "", "produto": "",
        "obs": obs_msg, "cidade": "", "confianca": "baixa"
    }


def avaliar(row):
    titulo  = str(row.get("titulo",  ""))
    veiculo = str(row.get("veiculo", ""))
    empresa = str(row.get("empresa", ""))
    midia   = str(row.get("midia",   ""))

    if is_paywall(veiculo):
        r = _vazio_avaliacao(f"PAYWALL — leitura incompleta. Acessar {veiculo} manualmente.")
        return r

    texto = raspar_texto(row.get("url_fonte", "") or row.get("link", ""))
    sem_texto = not texto
    conteudo = texto or f"[Texto indisponível — avalie pelo título: {titulo}]"

    prompt = PROMPT_AVALIACAO.format(
        empresa=empresa, titulo=titulo, veiculo=veiculo,
        tipo_midia=midia, conteudo=conteudo
    )

    for tentativa in range(3):
        try:
            msg = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=900,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = msg.content[0].text.strip()
            raw = re.sub(r"```json\s*", "", raw)
            raw = re.sub(r"```\s*", "", raw)
            res = json.loads(raw)
            # Rename "data" (Dado Proativo) to avoid conflict with date column
            res["dado_proativo"] = res.pop("data", False)
            if sem_texto:
                obs = res.get("obs", "")
                res["obs"] = "; ".join(filter(None, [obs, "SEM_TEXTO — não foi possível acessar o link"]))
                res["confianca"] = "baixa"
            return res
        except json.JSONDecodeError:
            time.sleep(2)
        except Exception as e:
            if tentativa < 2:
                time.sleep(2 ** (tentativa + 1))
            else:
                return _vazio_avaliacao(f"ERRO — {str(e)[:120]}")


print(f"🤖 Avaliando {len(df_main)} matérias com Claude ({CLAUDE_MODEL})...")
print(f"   Estimativa: ~{max(1, len(df_main)*5//60)}–{max(2, len(df_main)*8//60)} minutos\n")

avaliacoes = []
for i, (_, row) in enumerate(df_main.iterrows()):
    print(f"   [{i+1}/{len(df_main)}] {str(row.get('titulo',''))[:65]}...")
    avaliacoes.append(avaliar(row))
    time.sleep(0.5)

df_av = pd.DataFrame(avaliacoes)
df_main = pd.concat([df_main.reset_index(drop=True), df_av], axis=1)

n_humano = int(df_main.apply(
    lambda r: any(str(r.get(f"criterio_{c}", "")) == "ver humano"
                  for c in [11, 12, 14, 15, 16])
              or "PAYWALL" in str(r.get("obs", ""))
              or "ERRO" in str(r.get("obs", "")),
    axis=1
).sum())

print(f"\n✅ AVALIAÇÃO CONCLUÍDA")
print(f"   Destaques:       {(df_main['protagonismo']=='Destaque').sum()}")
print(f"   Menções:         {(df_main['protagonismo']=='Menção').sum()}")
print(f"   Negativos:       {df_main['protagonismo'].str.contains('Negativo', na=False).sum()}")
print(f"   Revisão humana:  {n_humano}")


In [ ]:
# ============================================================
# CÉLULA 6 — Gerar planilha .xlsx processada para download
# ============================================================

print("📊 Gerando planilha processada...")

uploaded_clip_bytes = list(uploaded.values())[0]
wb_orig = openpyxl.load_workbook(BytesIO(uploaded_clip_bytes))
ws_orig = wb_orig["Matérias"] if "Matérias" in wb_orig.sheetnames else wb_orig.active

wb_out = openpyxl.Workbook()
wb_out.remove(wb_out.active)

# ── ABA 1: "Cópia de Matérias" (backup idêntico) ─────────────
ws_copia = wb_out.create_sheet("Cópia de Matérias")
for row in ws_orig.iter_rows():
    for cell in row:
        ws_copia[cell.coordinate] = cell.value
print("   ✅ Aba 'Cópia de Matérias' criada")

# ── ABA 2: "Matérias" com linhas pintadas de vermelho ────────
ws_mat = wb_out.create_sheet("Matérias")
for row in ws_orig.iter_rows():
    for cell in row:
        ws_mat[cell.coordinate] = cell.value

LINHA_INICIO_DADOS = 5
titulos_para_pintar = set(df[df["motivo_vermelho"] != ""]["titulo"].astype(str).str.strip())
linhas_pintadas = 0
for xl_row in ws_mat.iter_rows(min_row=LINHA_INICIO_DADOS):
    titulo_celula = str(xl_row[0].value or "").strip()
    if titulo_celula in titulos_para_pintar:
        for cell in xl_row:
            cell.fill = FILL_VERMELHO
        linhas_pintadas += 1
print(f"   ✅ Aba 'Matérias' criada — {linhas_pintadas} linhas pintadas de vermelho")

# ── ABA 3: "Gabarito Dashboard" — 30 colunas A-AD ─────────────
ws_gab = wb_out.create_sheet("Gabarito Dashboard")

HEADERS_GAB = [
    "Índice",                                          # A
    "Mês",                                             # B
    "Título",                                          # C
    "Veículo",                                         # D
    "Data",                                            # E
    "Link",                                            # F
    "Empresa",                                         # G
    "Protagonismo (com base nos crit 1 a 7, decide)",  # H
    "Conta pra gente qual foi o critério",             # I
    "8",                                               # J — Protagonismo Editorial
    "9",                                               # K — Volume de Menções
    "10",                                              # L — CTA/Link
    "11",                                              # M — Recursos Visuais
    "12",                                              # N — Gráficos/Tabelas
    "13",                                              # O — Páginas Impressas
    "14",                                              # P — Capas e Redes Sociais
    "15",                                              # Q — Audiovisual
    "16",                                              # R — Análise Especial
    "soma (destaque + bonus)",                         # S
    "PR Tec+Produto",                                  # T
    "PR Puro Imobis",                                  # U
    "Data",                                            # V — Dado Proativo
    "Data Carona",                                     # W
    "Retranca.- humano",                               # X
    "Tema",                                            # Y
    "Subtema",                                         # Z
    "Produto",                                         # AA
    "OBS",                                             # AB
    "Cidade",                                          # AC
    "Estado",                                          # AD
]

ws_gab.append(HEADERS_GAB)
for cell in ws_gab[1]:
    cell.fill = FILL_HEADER
    cell.font = FONT_HEADER


def _v(row, key, default=""):
    v = row.get(key, default)
    if v is None:
        return default
    if isinstance(v, float) and str(v) == "nan":
        return default
    return v


for i, (_, row) in enumerate(df_main.iterrows(), 1):
    titulo  = str(_v(row, "titulo"))
    veiculo = str(_v(row, "veiculo"))
    link    = str(_v(row, "link"))

    tem_revisao = (
        any(str(_v(row, f"criterio_{c}")) == "ver humano" for c in [11, 12, 14, 15, 16])
        or "PAYWALL" in str(_v(row, "obs"))
        or "ERRO" in str(_v(row, "obs"))
    )

    linha = [
        i,                                                      # A  Índice
        _v(row, "mes"),                                         # B  Mês
        titulo,                                                 # C  Título
        veiculo,                                                # D  Veículo
        _v(row, "data", DATA_CLIPPING),                         # E  Data
        link,                                                   # F  Link
        str(_v(row, "empresa")),                                # G  Empresa
        str(_v(row, "protagonismo", "Menção")),                 # H  Protagonismo
        str(_v(row, "criterio_ativado", "nenhum")),             # I  Critério ativado
        _v(row, "criterio_8",  0),                              # J  8
        _v(row, "criterio_9",  0),                              # K  9
        _v(row, "criterio_10", 0),                              # L  10
        _v(row, "criterio_11", 0),                              # M  11
        _v(row, "criterio_12", 0),                              # N  12
        _v(row, "criterio_13", 0),                              # O  13
        _v(row, "criterio_14", "ver humano"),                   # P  14
        _v(row, "criterio_15", 0),                              # Q  15
        _v(row, "criterio_16", "ver humano"),                   # R  16
        _v(row, "soma", 0),                                     # S  soma
        "TRUE" if row.get("pr_tec_produto") else "FALSE",       # T  PR Tec+Produto
        "TRUE" if row.get("pr_puro_imobis") else "FALSE",       # U  PR Puro Imobis
        "TRUE" if row.get("dado_proativo")  else "FALSE",       # V  Data (Dado Proativo)
        "TRUE" if row.get("data_carona")    else "FALSE",       # W  Data Carona
        "",                                                     # X  Retranca (humano)
        str(_v(row, "tema")),                                   # Y  Tema
        str(_v(row, "subtema")),                                # Z  Subtema
        str(_v(row, "produto")),                                # AA Produto
        str(_v(row, "obs")),                                    # AB OBS
        str(_v(row, "cidade")),                                 # AC Cidade
        str(_v(row, "estado")),                                 # AD Estado
    ]
    ws_gab.append(linha)

    if tem_revisao:
        xl_num = i + 1
        for c in range(1, len(HEADERS_GAB) + 1):
            ws_gab.cell(row=xl_num, column=c).fill = FILL_AMARELO

ws_gab.column_dimensions["C"].width = 60
ws_gab.column_dimensions["F"].width = 45
ws_gab.column_dimensions["AB"].width = 55
print(f"   ✅ Aba 'Gabarito Dashboard' criada — {len(df_main)} linhas")

# ── ABA 4: "Double Check" ──────────────────────────────────────
if len(df_double_check) > 0:
    ws_dc = wb_out.create_sheet("Double Check")
    ws_dc.append(["Título", "Veículo", "Data", "Canal", "Empresa", "Link", "OBS"])
    for _, r in df_double_check.iterrows():
        ws_dc.append([
            str(r.get("titulo", "")), str(r.get("veiculo", "")),
            str(r.get("data", DATA_CLIPPING)), str(r.get("canal", "")),
            str(r.get("empresa", "")), str(r.get("link", "")),
            "Verificar contexto — possível off-topic"
        ])
    print(f"   ✅ Aba 'Double Check' criada — {len(df_double_check)} linhas")

# ── ABA 5: "Totais" ────────────────────────────────────────────
ws_tot = wb_out.create_sheet("Totais")
EMP = ["Loft", "QuintoAndar", "ZAP", "Superlógica"]
pts  = {e: 0  for e in EMP}
dests = {e: [] for e in EMP}

for _, r in df_main.iterrows():
    if r.get("protagonismo") == "Destaque" and r.get("empresa") in pts:
        pts[r["empresa"]] += 1  # count of Destaques per CONTEXTO section 6
        dests[r["empresa"]].append(r)

por_midia  = df_main["midia"].value_counts().to_dict()
por_estado = df_main["estado"].value_counts().head(10).to_dict()
n_humano_tot = int(df_main.apply(
    lambda r: (
        any(str(r.get(f"criterio_{c}", "")) == "ver humano" for c in [11, 12, 14, 15, 16])
        or "PAYWALL" in str(r.get("obs", ""))
        or "ERRO"    in str(r.get("obs", ""))
    ),
    axis=1
).sum())

ws_tot.append(["PR Score — Totais do Dia"])
ws_tot.append([])
ws_tot.append(["EMPRESA", "Pontos", "Destaques"])
for e in EMP:
    ws_tot.append([e, pts[e], len(dests[e])])
ws_tot.append([])
ws_tot.append(["TIPO DE MÍDIA", "Qtd"])
for k, v in sorted(por_midia.items(), key=lambda x: -x[1]):
    ws_tot.append([k, v])
ws_tot.append([])
ws_tot.append(["ESTADO (top 10)", "Qtd"])
for k, v in sorted(por_estado.items(), key=lambda x: -x[1]):
    ws_tot.append([k, v])
ws_tot.append([])
ws_tot.append(["RESUMO", ""])
ws_tot.append(["Total avaliadas",   len(df_main)])
ws_tot.append(["Pintadas vermelho", n_vermelho])
ws_tot.append(["Revisão humana",    n_humano_tot])
print("   ✅ Aba 'Totais' criada")

data_arquivo = DATA_CLIPPING.replace("/", "-")
nome_arquivo = f"PR_Score_{data_arquivo}.xlsx"
wb_out.save(nome_arquivo)
print(f"\n📥 Iniciando download de '{nome_arquivo}'...")
files.download(nome_arquivo)
print("✅ Planilha gerada com sucesso!")


In [ ]:
# ============================================================
# CÉLULA 7 — Revisão humana + WhatsApp + HTML do e-mail
# ============================================================
# Antes de gerar os outputs finais, o humano decide:
#   - Thumbs UP  (1): usar a planilha gerada automaticamente
#   - Thumbs DOWN (2): fazer upload da planilha revisada manualmente

print("=" * 60)
print("👤 ETAPA DE REVISÃO HUMANA")
print("=" * 60)
print()

if n_humano > 0:
    print(f"⚠️  {n_humano} matéria(s) precisam de olhar humano (marcadas em amarelo).")
else:
    print("✅  Nenhuma matéria exige revisão humana obrigatória.")

print()
print("Escolha como prosseguir:")
print("  1 — 👍 Aprovar resultados automáticos e gerar WhatsApp + e-mail")
print("  2 — 👎 Fazer upload da planilha revisada pelo humano")
print()

opcao = input("Digite 1 ou 2 e pressione Enter: ").strip()

if opcao == "2":
    print()
    print("📂 Faça upload da planilha revisada (.xlsx):")
    uploaded_rev = files.upload()
    rev_filename = list(uploaded_rev.keys())[0]
    wb_rev = openpyxl.load_workbook(BytesIO(uploaded_rev[rev_filename]))
    ws_rev = wb_rev["Gabarito Dashboard"] if "Gabarito Dashboard" in wb_rev.sheetnames else wb_rev.active
    headers_rev = [str(c.value or "") for c in ws_rev[1]]

    def _idx(nome):
        try: return headers_rev.index(nome)
        except: return -1

    col_protagonismo = _idx("Protagonismo (com base nos crit 1 a 7, decide)")
    col_empresa      = _idx("Empresa")
    col_titulo       = _idx("Título")
    col_veiculo      = _idx("Veículo")
    col_data         = _idx("Data")
    col_link         = _idx("Link")
    col_soma         = _idx("soma (destaque + bonus)")

    EMP   = ["Loft", "QuintoAndar", "ZAP", "Superlógica"]
    pts   = {e: 0  for e in EMP}
    dests = {e: [] for e in EMP}

    for xl_row in ws_rev.iter_rows(min_row=2, values_only=True):
        prot    = str(xl_row[col_protagonismo] or "") if col_protagonismo >= 0 else ""
        empresa = str(xl_row[col_empresa]      or "") if col_empresa      >= 0 else ""
        if prot == "Destaque" and empresa in pts:
            soma_val = xl_row[col_soma] if col_soma >= 0 else None
            try:
                pts[empresa] += int(soma_val or 1)
            except:
                pts[empresa] += 1
            dests[empresa].append({
                "titulo":    xl_row[col_titulo]  if col_titulo  >= 0 else "",
                "veiculo":   xl_row[col_veiculo] if col_veiculo >= 0 else "",
                "data":      xl_row[col_data]    if col_data    >= 0 else "",
                "link":      xl_row[col_link]    if col_link    >= 0 else "",
                "soma":      soma_val,
            })
    print()
    print("✅ Planilha revisada carregada com sucesso!")

else:
    EMP = ["Loft", "QuintoAndar", "ZAP", "Superlógica"]
    # pts and dests already built in Cell 6

print()
print("=" * 60)
print("📱 Gerando mensagem WhatsApp e HTML do e-mail...")
print("=" * 60)

CORES  = {"Loft": "#FF6B35", "QuintoAndar": "#1A73E8", "ZAP": "#34A853", "Superlógica": "#212121"}
EMOJIS = {"Loft": "🟠",      "QuintoAndar": "🔵",      "ZAP": "🟢",      "Superlógica": "⚫"}

# ── WhatsApp ──────────────────────────────────────────────────
wa = f"📊 PR Score — {DATA_CLIPPING}\n\n"
for e in EMP:
    wa += f"{EMOJIS[e]} {e}: {pts[e]} pt{'s' if pts[e] != 1 else ''}\n"

if dests["Loft"]:
    wa += "\n📰 Destaques Loft do dia:\n"
    for d in dests["Loft"]:
        wa += f"• {d.get('titulo','')} - {d.get('veiculo','')} {d.get('link','')}\n"

print()
print("=" * 60)
print("📱 MENSAGEM WHATSAPP — copie e cole no grupo")
print("=" * 60)
print(wa)

# ── HTML do e-mail ────────────────────────────────────────────
def secao(empresa):
    cor = CORES[empresa]
    ds  = dests.get(empresa, [])
    if not ds:
        itens = '<p style="color:#888;font-style:italic">Sem destaques hoje.</p>'
    else:
        itens = "".join(
            f'<div style="border-left:3px solid {cor};padding:10px 16px;margin-bottom:10px;background:#fafafa">'
            f'<a href="{d.get("link","#")}" style="color:#111;font-weight:600;font-size:15px;text-decoration:none">{d.get("titulo","")}</a>'
            f'<div style="margin-top:4px;font-size:13px;color:#666">'
            f'{d.get("veiculo","")} · {d.get("data","")}'
            f'<span style="background:{cor};color:#fff;padding:2px 8px;border-radius:12px;font-size:11px;margin-left:6px">'
            f'{int(d.get("soma") or 1)} pt{"s" if int(d.get("soma") or 1) != 1 else ""}'
            f'</span></div></div>'
            for d in ds
        )
    return (
        f'<div style="margin-bottom:28px">'
        f'<h2 style="color:{cor};border-bottom:2px solid {cor};padding-bottom:6px">'
        f'{EMOJIS[empresa]} {empresa}'
        f'<span style="font-size:14px;font-weight:normal;color:#888"> — {pts[empresa]} pontos</span>'
        f'</h2>{itens}</div>'
    )

placar = " ".join(
    f'<div style="text-align:center;padding:10px 16px">'
    f'<div style="font-size:22px;font-weight:700;color:{CORES[e]}">{pts[e]}</div>'
    f'<div style="font-size:12px;color:#666">{e}</div></div>'
    for e in EMP
)

html = (
    f'<!DOCTYPE html><html lang="pt-BR"><head><meta charset="UTF-8">'
    f'<meta name="viewport" content="width=device-width,initial-scale=1.0">'
    f'<title>PR Score — {DATA_CLIPPING}</title></head>'
    f'<body style="font-family:Arial,sans-serif;max-width:660px;margin:0 auto;padding:24px;color:#222">'
    f'<div style="background:#FF6B35;padding:20px 24px;border-radius:8px;margin-bottom:20px">'
    f'<h1 style="color:#fff;margin:0;font-size:22px">📊 PR Score</h1>'
    f'<p style="color:#fff;margin:4px 0 0;opacity:.9">{DATA_CLIPPING}</p></div>'
    f'<div style="display:flex;justify-content:space-around;background:#f5f5f5;border-radius:8px;margin-bottom:28px">'
    f'{placar}</div>'
    + "".join(secao(e) for e in EMP) +
    f'<div style="border-top:1px solid #eee;padding-top:14px;font-size:12px;color:#999;text-align:center">'
    f'PR Score · Loft · Time de PR</div></body></html>'
)

nome_html = f"email_pr_score_{DATA_CLIPPING.replace('/', '_')}.html"
with open(nome_html, "w", encoding="utf-8") as f:
    f.write(html)
files.download(nome_html)

print()
print("=" * 60)
print(f"📧 HTML baixado: {nome_html}")
print("=" * 60)
print()
print("✅ TUDO PRONTO! Próximos passos:")
print("   1. Revisar planilha — linhas vermelhas (excluídas) e amarelas (revisão humana)")
print("   2. Copiar Gabarito Dashboard para a planilha oficial do Drive")
print("   3. Preencher coluna Retranca consultando o Monday")
print("   4. Enviar WhatsApp para o grupo")
print("   5. Após aprovação: colar HTML no RD Station e enviar")
